# Recommendation Engine

**Deliverable 04** - Part 2

**Author:** Abdelrahman Yasser Hassan Zaky  
**ID:** 231000102

## Step 1: Load Best Model & Encoders

In [ ]:
import pickle
import pandas as pd
import numpy as np

with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

with open('../data/model_comparison_results.pkl', 'rb') as f:
    results = pickle.load(f)

best_model_name = max(results, key=lambda m: results[m]['f1'])
print(f'Best model: {best_model_name}')

with open(f'../data/{best_model_name.lower().replace(" ", "_")}_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

df = pd.read_csv('../data/workout_dataset.csv')
le_type = data['le_type']
le_muscle = data['le_muscle']
le_equipment = data['le_equipment']
le_difficulty = data['le_difficulty']

print(f'Dataset loaded: {len(df)} exercises')

## Step 2: Create Recommendation Function

In [ ]:
def recommend_exercises(user_level, user_goal, available_equipment, top_n=10):
    filtered = df[df['difficulty'] == user_level].copy()
    filtered = filtered[filtered['equipment'].isin(available_equipment)]
    
    if user_goal == 'cardio':
        goal_types = ['cardio', 'compound']
    elif user_goal == 'strength':
        goal_types = ['push', 'pull', 'isolation', 'legs']
    else:
        goal_types = list(df['type'].unique())
    
    filtered = filtered[filtered['type'].isin(goal_types)]
    
    if len(filtered) > 0:
        return filtered.head(top_n)
    
    fallback = df[df['difficulty'] == user_level].copy()
    fallback = fallback[fallback['type'].isin(goal_types)]
    return fallback.head(top_n)

## Step 3: Test with Example Users

In [ ]:
print('=' * 70)
print('TEST CASE 1: Beginner, Cardio, Bodyweight Only')
print('=' * 70)
recs = recommend_exercises('beginner', 'cardio', ['bodyweight'])
for _, row in recs.iterrows():
    print(f'  {row["name"]:<35} | {row["target_muscles"]:<15} | {row["type"]:<10} | {row["description"]}')

In [ ]:
print('=' * 70)
print('TEST CASE 2: Intermediate, Strength, Full Gym')
print('=' * 70)
recs = recommend_exercises('intermediate', 'strength', ['barbell', 'dumbbells', 'machine', 'cable machine'])
for _, row in recs.iterrows():
    print(f'  {row["name"]:<35} | {row["target_muscles"]:<15} | {row["type"]:<10} | {row["description"]}')

In [ ]:
print('=' * 70)
print('TEST CASE 3: Advanced, Hybrid, Home Gym')
print('=' * 70)
recs = recommend_exercises('advanced', 'hybrid', ['dumbbells', 'bodyweight', 'resistance band'])
for _, row in recs.iterrows():
    print(f'  {row["name"]:<35} | {row["target_muscles"]:<15} | {row["type"]:<10} | {row["description"]}')

## Step 4: Save Recommendation Engine

In [ ]:
engine = {
    'model': best_model,
    'le_type': le_type,
    'le_muscle': le_muscle,
    'le_equipment': le_equipment,
    'le_difficulty': le_difficulty,
    'df': df,
}

with open('../data/recommendation_engine.pkl', 'wb') as f:
    pickle.dump(engine, f)
print('Recommendation engine saved to: ../data/recommendation_engine.pkl')